# 03 – Очистка и подготовка данных Европы

## Шаг 1 – Диагностика представлений пропусков

На этом шаге только читаем исходную базу и показываем результаты. Преобразование значений, заполнение пропусков, исключение строк и создание очищенной базы требуют отдельного согласования. Проверяем всю таблицу блоками, без загрузки полной базы в память.

In [1]:
# Подключаем библиотеки для чтения Parquet, таблиц и проверки целостности
from pathlib import Path
import hashlib
import json
from collections import Counter
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

# Находим корень проекта независимо от папки запуска ноутбука
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'europe/data/source/osn23_landings_merged.parquet.gzip').exists())
SOURCE = ROOT / 'europe/data/source/osn23_landings_merged.parquet.gzip'

# Считаем SHA256 блоками: весь исходный файл не нужно загружать в память
def source_hash():
    h = hashlib.sha256()
    with SOURCE.open('rb') as stream:
        for block in iter(lambda: stream.read(4 * 1024**2), b''):
            h.update(block)
    return h.hexdigest()

# Сверяем источник и число строк с уже выполненным аудитом
before = source_hash()
audit = json.loads((ROOT / 'europe/reports/profile_summary.json').read_text(encoding='utf8'))
assert before == audit['source_sha256'], 'Источник отличается от файла аудита 01.'
parquet = pq.ParquetFile(SOURCE)
assert parquet.metadata.num_rows == audit['rows']
print('Записей:', f'{parquet.metadata.num_rows:,}'.replace(',', ' '))
print('SHA256 совпадает с аудитом 01.')

Записей: 7 984 254
SHA256 совпадает с аудитом 01.


## Подсчет и примеры

Разделяем физический `null`, точную пустую строку `""` и непустую строку, состоящую только из пробельных символов. Проверяем строковые столбцы верхнего уровня; содержимое списков `ILS` и `attempt_times` здесь не анализируется. Примеры ограничены тремя записями на поле и вид пропуска. Позиция строки – нумерация с нуля в исходном файле, а не идентификатор полета.

Пробелы проверяются диагностически через `str.strip()`. Исходные значения не заменяются и не обрезаются. Текстовые обозначения вроде `unknown` не признаются пропусками автоматически.

In [2]:
# Выбираем строковые поля верхнего уровня; вложенные списки здесь не проверяем
fields = [f.name for f in parquet.schema_arrow
          if pa.types.is_string(f.type) or pa.types.is_large_string(f.type)]
counts = {field: Counter() for field in fields}
examples = []
shown = Counter()
# Читаем по 100 000 строк и сохраняем глобальную позицию примеров
offset = 0
for batch in parquet.iter_batches(batch_size=100_000, columns=fields):
    frame = batch.to_pandas(ignore_metadata=True)
    for field in fields:
        values = frame[field]
        # Различаем null, точную пустую строку и строку из пробелов
        masks = {
            'null': values.isna(),
            'empty_string': values.eq('').fillna(False),
            'whitespace_only': (values.notna() & values.ne('')
                                & values.str.strip().eq('')).fillna(False),
        }
        for kind, mask in masks.items():
            counts[field][kind] += int(mask.sum())
            # Ограничиваем примеры тремя строками на поле и вид пропуска
            remaining = 3 - shown[(field, kind)]
            if remaining > 0:
                for position in frame.index[mask][:remaining]:
                    examples.append({'field': field, 'kind': kind,
                                     'source_row_position': offset + int(position),
                                     'raw_repr': repr(values.loc[position])})
                    shown[(field, kind)] += 1
    offset += len(frame)

# Проверяем полный охват файла и рассчитываем доли от всех строк
assert offset == audit['rows']
missingness = pd.DataFrame([{'field': field, **dict(counts[field])} for field in fields])
for kind in ['null', 'empty_string', 'whitespace_only']:
    missingness[kind + '_percent'] = 100 * missingness[kind] / offset
assert source_hash() == before, 'Контрольная сумма источника изменилась.'
display(missingness)
print('Исходный файл не изменился. Преобразования не выполнялись.')

,field,null,empty_string,whitespace_only,null_percent,empty_string_percent,whitespace_only_percent
0,flight_id,0,0,0,0.000000,0.000000,0.0
1,callsign,0,0,0,0.000000,0.000000,0.0
2,icao24,0,0,0,0.000000,0.000000,0.0
3,registration,859077,242227,0,10.759640,3.033809,0.0
4,typecode,859077,475015,0,10.759640,5.949397,0.0
5,icaoaircrafttype,859077,543159,0,10.759640,6.802877,0.0
6,airport,0,0,0,0.000000,0.000000,0.0
7,country,0,0,0,0.000000,0.000000,0.0
8,market_segment,1515276,0,0,18.978304,0.000000,0.0
9,AC_CLASS,1652382,0,0,20.695509,0.000000,0.0


Исходный файл не изменился. Преобразования не выполнялись.


In [3]:
# Показываем исходные представления пропусков для обсуждения правил
display(pd.DataFrame(examples))
print('Примеры показывают исходные представления; это не правила обработки.')

,field,kind,source_row_position,raw_repr
0,registration,null,1,nan
1,registration,null,2,nan
2,registration,null,29,nan
3,registration,empty_string,0,''
4,registration,empty_string,7,''
5,registration,empty_string,9,''
6,typecode,null,1,nan
7,typecode,null,2,nan
8,typecode,null,29,nan
9,typecode,empty_string,0,''


Примеры показывают исходные представления; это не правила обработки.


## Шаг 2 – Согласованная нормализация пустых строк

Согласовано: только в `registration`, `typecode` и `icaoaircrafttype` заменяем точную пустую строку на физический null. Сохраняем три булевых поля `<имя>_was_empty_string`: True означает исходную пустую строку, False – любое другое исходное значение, включая исходный null. Вместе с неизмененным источником это позволяет восстановить исходное представление.

Записи и остальные значения сохраняются. Это промежуточная рабочая копия после одного правила, а не окончательная аналитическая база. Заполнение пропусков и другие правила еще не согласованы.

In [4]:
# Подключаем операции Arrow для преобразования без потери типов
import pyarrow.compute as pc

# Нормализуем только три согласованных поля; создаем отдельную рабочую копию
approved_fields = ['registration', 'typecode', 'icaoaircrafttype']
WORK = ROOT / 'europe/data/working'
WORK.mkdir(parents=True, exist_ok=True)
DESTINATION = WORK / '01_empty_strings_normalized.parquet'
TEMPORARY = WORK / '01_empty_strings_normalized.parquet.part'
output_schema = parquet.schema_arrow
for field in approved_fields:
    assert field + '_was_empty_string' not in output_schema.names
    output_schema = output_schema.append(pa.field(field + '_was_empty_string', pa.bool_()))

replaced = Counter()
written = 0
# Сначала пишем временный файл; для каждой замены сохраняем обратимый флаг
with pq.ParquetWriter(TEMPORARY, output_schema, compression='gzip', compression_level=1) as writer:
    for batch in parquet.iter_batches(batch_size=100_000):
        table = pa.Table.from_batches([batch])
        for field in approved_fields:
            original = table[field]
            mask = pc.fill_null(pc.equal(original, ''), False)
            replaced[field] += pc.sum(pc.cast(mask, pa.int64())).as_py()
            normalized = pc.if_else(mask, pa.scalar(None, type=original.type), original)
            table = table.set_column(table.schema.get_field_index(field),
                                     table.schema.field(field), normalized)
            table = table.append_column(field + '_was_empty_string', mask)
        writer.write_table(table)
        written += len(table)

# Проверяем обратимость каждого изменения и неизменность остальных полей
candidate = pq.ParquetFile(TEMPORARY)
assert candidate.metadata.num_rows == written == audit['rows']
checked = 0
after_nulls = Counter()
for source_batch, result_batch in zip(parquet.iter_batches(batch_size=100_000),
                                      candidate.iter_batches(batch_size=100_000), strict=True):
    original = pa.Table.from_batches([source_batch])
    result = pa.Table.from_batches([result_batch])
    for field in approved_fields:
        flag = result[field + '_was_empty_string']
        expected_flag = pc.fill_null(pc.equal(original[field], ''), False)
        assert flag.equals(expected_flag)
        assert pc.sum(pc.cast(pc.fill_null(pc.equal(result[field], ''), False), pa.int64())).as_py() == 0
        after_nulls[field] += result[field].null_count
        restored = pc.if_else(flag, pa.scalar('', type=original[field].type), result[field])
        result = result.set_column(result.schema.get_field_index(field), original.schema.field(field), restored)
    assert result.select(original.column_names).equals(original)
    checked += len(original)
assert checked == written
assert source_hash() == before
for field in approved_fields:
    assert replaced[field] == counts[field]['empty_string']
    assert after_nulls[field] == counts[field]['null'] + replaced[field]

# После всех проверок публикуем рабочую копию заменой временного файла
candidate.close()
TEMPORARY.replace(DESTINATION)
change_log = pd.DataFrame([{'field': field, 'empty_strings_replaced': replaced[field],
                            'null_before': counts[field]['null'], 'null_after': after_nulls[field]}
                           for field in approved_fields])
display(change_log)
# Сохраняем счетчики и контрольные суммы в журнал воспроизводимости
REPORT = ROOT / 'europe/reports/preparation'
REPORT.mkdir(parents=True, exist_ok=True)
change_log.to_csv(REPORT / '01_empty_strings_normalized.csv', index=False)
(REPORT / '01_empty_strings_normalized.json').write_text(json.dumps({
    'rule': 'exact empty string to null in three approved fields',
    'approved_fields': approved_fields,
    'source_sha256': before, 'source_unchanged': True,
    'output': str(DESTINATION.relative_to(ROOT)),
    'output_sha256': hashlib.file_digest(DESTINATION.open('rb'), 'sha256').hexdigest(),
    'rows_before': written, 'rows_after': written, 'rows_deleted': 0,
    'reversible_comparison_passed': True,
    'changes': change_log.to_dict(orient='records'),
}, ensure_ascii=False, indent=2) + '\n', encoding='utf8')
print('Сохранена рабочая копия:', DESTINATION)
print('Все записи сохранены. Проверена обратимость и неизменность остальных полей.')

,field,empty_strings_replaced,null_before,null_after
0,registration,242227,859077,1101304
1,typecode,475015,859077,1334092
2,icaoaircrafttype,543159,859077,1402236


Сохранена рабочая копия: D:\AI-ML Engineering\Projects\5 - Approach Risk ML\Approach Risk\europe\data\working\01_empty_strings_normalized.parquet
Все записи сохранены. Проверена обратимость и неизменность остальных полей.


## Шаг 3 – Диагностика неоднозначных кодов самолетов

Читаем рабочую копию после согласованной нормализации пустых строк. Сравниваем `icaoaircrafttype` с преобладающим шаблоном `[A-Z][0-9][A-Z]` и с `typecode`. Это структурная проверка, а не справочник допустимых авиационных кодов. Верхний регистр используется только при сравнении, исходные значения не изменяются.

Показываем все непустые значения вне шаблона, пары с `typecode`, распределение по аэропортам и до трех исходных примеров на значение. Возможные соответствия внутри базы служат подсказкой для последующей проверки источника.

In [1]:
# Готовим диагностику классификаций самолетов в нормализованной копии
from pathlib import Path
from collections import Counter
import hashlib
import json
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'europe/data/source/osn23_landings_merged.parquet.gzip').exists())
WORKING = ROOT / 'europe/data/working/01_empty_strings_normalized.parquet'
REPORT = ROOT / 'europe/reports/preparation'
log = json.loads((REPORT / '01_empty_strings_normalized.json').read_text(encoding='utf8'))

def hash_file(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()

# Проверяем, что вход соответствует журналу предыдущего шага
working_before = hash_file(WORKING)
assert working_before == log['output_sha256']
reader = pq.ParquetFile(WORKING)
pair_counts = Counter()
pair_events = Counter()
airport_counts = Counter()
consistent_pairs = Counter()
sample_counts = Counter()
samples = []
rows = flagged = matched_typecode = missing_class = 0
columns = ['flight_id', 'icao24', 'landing_time', 'airport', 'typecode',
           'icaoaircrafttype', 'GoA', 'n_attempts']
# Сканируем всю таблицу и собираем сочетания кодов, метки и примеры
for batch in reader.iter_batches(batch_size=100_000, columns=columns):
    frame = batch.to_pandas(ignore_metadata=True)
    classification = frame['icaoaircrafttype']
    # Шаблон выделяет значения для разбора, но сам по себе не доказывает ошибку
    valid_shape = classification.str.fullmatch(r'[A-Z][0-9][A-Z]', na=False)
    suspicious = classification.notna() & ~valid_shape
    missing_class += int(classification.isna().sum())
    matches = (classification.notna() & frame['typecode'].notna()
               & classification.str.upper().eq(frame['typecode'].str.upper())).fillna(False)
    matched_typecode += int(matches.sum())
    for record in frame.loc[valid_shape & frame['typecode'].notna(), ['typecode', 'icaoaircrafttype']].itertuples(index=False):
        consistent_pairs[(record.typecode, record.icaoaircrafttype)] += 1
    for position, record in frame.loc[suspicious].iterrows():
        code = record['icaoaircrafttype']
        typecode = None if pd.isna(record['typecode']) else record['typecode']
        pair = (code, typecode)
        pair_counts[pair] += 1
        pair_events[pair] += int(record['GoA'])
        airport_counts[(code, record['airport'])] += 1
        if sample_counts[code] < 3:
            sample = record.to_dict()
            sample['source_row_position'] = rows + int(position)
            sample['equals_typecode_ignoring_case'] = bool(matches.loc[position])
            samples.append(sample)
            sample_counts[code] += 1
    flagged += int(suspicious.sum())
    rows += len(frame)

# Проверяем охват и неизменность входа, затем строим диагностические таблицы
assert rows == log['rows_after']
assert hash_file(WORKING) == working_before
pairs = pd.DataFrame([{'icaoaircrafttype': code, 'typecode': typecode, 'rows': count,
                       'go_arounds': pair_events[(code, typecode)]}
                      for (code, typecode), count in pair_counts.items()]).sort_values('rows', ascending=False)
codes = pairs.groupby('icaoaircrafttype')[['rows', 'go_arounds']].sum().sort_values('rows', ascending=False)
codes['ga_percent'] = 100 * codes['go_arounds'] / codes['rows']
airport_distribution = pd.DataFrame([{'icaoaircrafttype': code, 'airport': airport, 'rows': count}
                                    for (code, airport), count in airport_counts.items()]).sort_values('rows', ascending=False)
comparison = pd.DataFrame([{'typecode': typecode, 'icaoaircrafttype': code, 'rows': count}
                           for (typecode, code), count in consistent_pairs.items()])
affected_types = pairs['typecode'].dropna().unique()
comparison = comparison.loc[comparison['typecode'].isin(affected_types)].sort_values(['typecode', 'rows'], ascending=[True, False])
# Сохраняем отчеты; значения в рабочей базе не исправляем
codes.to_csv(REPORT / '02_aircraft_code_values.csv')
pairs.to_csv(REPORT / '02_aircraft_code_pairs.csv', index=False)
airport_distribution.to_csv(REPORT / '02_aircraft_code_airports.csv', index=False)
comparison.to_csv(REPORT / '02_aircraft_code_internal_reference.csv', index=False)
print('Всего записей:', rows)
print('Классификация отсутствует:', missing_class)
print('Непустые значения вне преобладающего шаблона:', flagged)
print('Совпадают с typecode без учета регистра:', matched_typecode)
display(codes)
print('Рабочая копия не изменена. Исправления не применялись.')

Всего записей: 7984254
Классификация отсутствует: 1402236
Непустые значения вне преобладающего шаблона: 6407
Совпадают с typecode без учета регистра: 130


,rows,go_arounds,ga_percent
icaoaircrafttype,,,
A332,1140,1,0.087719
a320,1092,8,0.732601
A21N,705,7,0.992908
GLF6,692,5,0.722543
FA8X,561,6,1.069519
B734,481,0,0.000000
82R,457,2,0.437637
C68A,357,0,0.000000
GL5T,330,0,0.000000


Рабочая копия не изменена. Исправления не применялись.


In [2]:
# Сопоставляем подозрительные коды с typecode и распределением по аэропортам
print('Пары значений вне шаблона и typecode:')
display(pairs)
print('Распределение по аэропортам:')
display(airport_distribution)
# Внутренние сочетания служат сравнением, а не официальным справочником
print('Другие классификации тех же typecode внутри базы – только для сравнения:')
display(comparison)
# Показываем ограниченное число исходных строк для ручной проверки
print('Примеры записей, до трех на значение классификации:')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(pd.DataFrame(samples))

Пары значений вне шаблона и typecode:


,icaoaircrafttype,typecode,rows,go_arounds
23,A332,NaN,1137,1
0,a320,NaN,1092,8
5,A21N,NaN,705,7
2,FA8X,Falcon 8X,561,6
6,GLF6,G650,495,5
12,B734,NaN,481,0
3,82R,737,457,2
1,C68A,680 Citation Latitude,357,0
4,GL5T,BD-700-1A11 Global 5000,330,0
8,GLF6,G650ER,197,0


Распределение по аэропортам:


,icaoaircrafttype,airport,rows
129,A332,LOWW,667
0,a320,LIRF,406
3,FA8X,LSGG,379
131,A332,EGSS,281
76,B734,LIMC,186
...,...,...,...
234,G-V,EHAM,1
236,G550,EDDM,1
237,A319,LFPG,1
239,F2TH,LIMC,1


Другие классификации тех же typecode внутри базы – только для сравнения:


,typecode,icaoaircrafttype,rows
171,G550,L2J,12
314,G650,L2J,1
224,G650ER,L2J,3


Примеры записей, до трех на значение классификации:


,flight_id,icao24,landing_time,airport,typecode,icaoaircrafttype,GoA,n_attempts,source_row_position,equals_typecode_ignoring_case
0,VLG442_25252,345643,2019-01-01 07:49:29+00:00,LIRF,NaN,a320,False,1,661,False
1,VLG65UY_25306,345643,2019-01-01 15:49:42+00:00,LIRF,NaN,a320,False,1,2990,False
2,VLG6137_25261,345643,2019-01-01 19:44:15+00:00,LIRF,NaN,a320,False,1,4392,False
3,ASJ794_7181,398275,2019-01-02 11:24:21+00:00,LSGG,680 Citation Latitude,C68A,False,1,7476,False
4,ASJ794_7182,398275,2019-01-02 16:31:36+00:00,LSGG,680 Citation Latitude,C68A,False,1,9266,False
5,LUC17_51290,4b1a17,2019-01-04 20:37:15+00:00,LSGG,Falcon 8X,FA8X,False,1,23249,False
6,BMS4H_59456,4a09ae,2019-01-05 17:56:33+00:00,ESSA,737,82R,False,1,28147,False
7,ULC91_72659,4d2188,2019-01-05 21:19:49+00:00,LSGG,BD-700-1A11 Global 5000,GL5T,False,1,29173,False
8,BMS6TT_146606,4a09ae,2019-01-06 16:23:53+00:00,LEMD,737,82R,False,1,33371,False
9,ASJ312_7668,39c8a9,2019-01-07 11:03:35+00:00,LSGG,680 Citation Latitude,C68A,False,1,37967,False


## Согласование следующего правила

Исправления кодов не применены. Выход за преобладающий шаблон и совпадение двух полей не доказывают конкретное правильное значение. Следующее правило согласуем после просмотра групп и проверки происхождения полей. Нормализация регистра, перенос кодов между полями, замена на null и удаление строк здесь не выполняются.

## Шаг 4 – Проверка по справочнику и предложения

Проверка источников оформлена в [отдельном разборе](../reports/preparation/03_aircraft_code_review.md). Использована публичная таблица [ICAO Doc 8643](https://www.icao.int/operational-safety/doc-8643-aircraft-type-designators/search), полученная 08.10.2026. Текущий справочник не доказывает историческое состояние в 2019–2023 годах или тип конкретного борта.

18 значений имеют прямое соответствие с одним Description: 5 888 записей и 27 отмеченных уходов. Еще пять значений остаются неразрешенными: 519 записей и 2 отмеченных ухода. Таблица ниже содержит предложения; они не применены.

In [1]:
# Читаем справочные предложения без автоматического применения
from pathlib import Path
import hashlib
import json
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'europe/data/source/osn23_landings_merged.parquet.gzip').exists())
REPORT = ROOT / 'europe/reports/preparation'
# Сохраняем текстовые значения CSV и сверяем размер набора предложений
proposals = pd.read_csv(REPORT / '03_aircraft_code_proposals.csv', keep_default_na=False)
assert len(proposals) == 23 and proposals['rows'].sum() == 6407
assert not proposals['applied'].any()
# Контрольная сумма подтверждает, что рабочая копия не менялась
with (ROOT / 'europe/data/working/01_empty_strings_normalized.parquet').open('rb') as stream:
    digest = hashlib.file_digest(stream, 'sha256').hexdigest()
log = json.loads((REPORT / '01_empty_strings_normalized.json').read_text(encoding='utf8'))
assert digest == log['output_sha256']
# Показываем источник, описание и статус каждого предложения
with pd.option_context('display.max_rows', None):
    display(proposals[['source_icaoaircrafttype', 'rows', 'go_arounds',
                       'lookup_designator', 'reference_description', 'status']])
print('Рабочая копия не изменена. Справочные предложения не применены.')

,source_icaoaircrafttype,rows,go_arounds,lookup_designator,reference_description,status
0,A332,1140,1,A332,L2J,direct_unique_reference_match
1,a320,1092,8,A320,L2J,direct_unique_reference_match
2,A21N,705,7,A21N,L2J,direct_unique_reference_match
3,GLF6,692,5,GLF6,L2J,direct_unique_reference_match
4,FA8X,561,6,FA8X,L3J,direct_unique_reference_match
5,B734,481,0,B734,L2J,direct_unique_reference_match
6,82R,457,2,82R,,unresolved
7,C68A,357,0,C68A,L2J,direct_unique_reference_match
8,GL5T,330,0,GL5T,L2J,direct_unique_reference_match
9,F2TH,137,0,F2TH,L2J,direct_unique_reference_match


Рабочая копия не изменена. Справочные предложения не применены.


## Требуется согласование справочного правила

Предлагаем добавить производное описание по прямым однозначным соответствиям справочника для 18 значений и признак источника сопоставления. Исходные `icaoaircrafttype` и `typecode` сохранить. Пять неразрешенных значений оставить для отдельного разбора. Перезапись исходных полей, удаление записей и заполнение typecode этим правилом не выполняются.

Только после согласования добавляется ячейка преобразования.

## Шаг 5 – Согласованное справочное обогащение

Согласовано добавление производного описания для 18 прямо сопоставленных значений. Исходные `icaoaircrafttype`, `typecode` и все остальные существующие поля сохраняются. Новая копия – `europe/data/working/02_aircraft_reference_enriched.parquet`.

Добавляем `aircraft_description_reference`, `aircraft_reference_status`, `aircraft_reference_source` и `aircraft_reference_checked_date`. Для 5 888 сопоставленных записей источник – публичная таблица [ICAO Doc 8643](https://www.icao.int/operational-safety/doc-8643-aircraft-type-designators/search), снимок 08.10.2026. Производное описание для остальных записей остается null: статус различает неразрешенные значения, отсутствующее исходное описание и значения вне текущего разбора.

Историческое соответствие справочника за 2019–2023 годы не подтверждено. Производное поле не означает независимое подтверждение типа конкретного борта. Сначала проверяем SHA256 входной копии и справочного снимка; после записи сравниваем все существующие поля с входом и проверяем производные значения.

In [1]:
# Запускаем согласованное справочное обогащение отдельным скриптом
from pathlib import Path
import runpy
import json
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'europe/data/source/osn23_landings_merged.parquet.gzip').exists())
# run_path загружает функции скрипта; main выполняет преобразование и проверки
module = runpy.run_path(str(ROOT / 'europe/scripts/apply_aircraft_reference.py'))
changes = module['main']()
display(changes)
# Показываем журнал: сохранность строк, меток и всех существующих полей
journal = json.loads((ROOT / 'europe/reports/preparation/04_aircraft_reference_applied.json').read_text(encoding='utf8'))
print('Записей до и после:', journal['rows_before'], journal['rows_after'])
print('Отмеченных уходов до и после:', journal['go_arounds_before'], journal['go_arounds_after'])
print('Справочное описание добавлено:', journal['rows_enriched'])
print('Статусы:', journal['status_counts'])
print('Все существующие поля неизменны:', journal['existing_columns_unchanged'])
print('Официальный источник:', journal['reference_url'])
print('Дата проверки:', journal['reference_checked_date'])

Rows preserved: 7984254 Rows enriched: 5888


,source_icaoaircrafttype,reference_description,rows_enriched,go_arounds
0,A21N,L2J,705,7
1,A319,L2J,5,0
2,A332,L2J,1140,1
3,AT76,L2T,25,0
4,B734,L2J,481,0
5,BE4W,L2J,68,0
6,C68A,L2J,357,0
7,E300,L1P,28,0
8,F2TH,L2J,137,0
9,FA7X,L3J,126,0


Записей до и после: 7984254 7984254
Отмеченных уходов до и после: 26253 26253
Справочное описание добавлено: 5888
Статусы: {'not_reviewed': 6575611, 'source_description_missing': 1402236, 'direct_unique_reference_match': 5888, 'unresolved': 519}
Все существующие поля неизменны: True
Официальный источник: https://doc8643.icao.int/External/AircraftTypes
Дата проверки: 2026-10-08


## Следующие действия требуют отдельного согласования

Выполнены только нормализация точных пустых строк в трех полях и согласованное справочное обогащение 18 значений. Исходные коды не перезаписаны. Пять неразрешенных значений не исправлены, записи не исключены. Следующий разбор и любое новое правило согласуются отдельно.

## Шаг 6 – Неразрешенные значения и история бортов

Только диагностика. Проверяем 519 записей с `82R`, `G550`, `G-V`, `GL6T`, `X` и все наблюдения тех же ICAO-адресов в рабочей копии после справочного обогащения. Регистрации и исходные коды сохраняются.

Официальные сведения, кандидаты и исторические ограничения оформлены в [отдельном разборе](../reports/preparation/05_unresolved_aircraft_review.md). Идентичность борта нельзя считать постоянной только по регистрации или адресу. Текущий реестр не заменяет историческое подтверждение.

In [1]:
# Исследуем оставшиеся неразрешенные классификации самолетов
from pathlib import Path
import runpy
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'europe/data/source/osn23_landings_merged.parquet.gzip').exists())
# Скрипт ищет другие сочетания тех же адресов во всей базе, не исправляя ее
module = runpy.run_path(str(ROOT / 'europe/scripts/diagnose_unresolved_aircraft.py'))
summary, history = module['main']()
# Сводка и история помогают оценить кандидатов на дальнейшую проверку
display(summary)
print('Другие сочетания полей тех же адресов во всей базе:')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(history)
print('ICAO-адресов:', history['icao24'].nunique())
print('Рабочая копия проверена по SHA256 и не изменена.')

icaoaircrafttype  rows  go_arounds  aircraft_addresses             first_landing              last_landing
             82R   457           2                   1 2019-01-05 17:56:33+00:00 2022-09-04 20:06:12+00:00
             G-V     8           0                   1 2019-03-19 11:02:24+00:00 2023-01-30 11:55:12+00:00
            G550    37           0                   3 2019-04-03 18:24:12+00:00 2023-04-06 10:34:33+00:00
            GL6T    12           0                   1 2021-05-17 10:55:18+00:00 2022-11-08 09:20:29+00:00
               X     5           0                   1 2019-11-29 12:31:39+00:00 2022-10-21 14:01:10+00:00


,icaoaircrafttype,rows,go_arounds,aircraft_addresses,first_landing,last_landing
0,82R,457,2,1,2019-01-05 17:56:33+00:00,2022-09-04 20:06:12+00:00
1,G-V,8,0,1,2019-03-19 11:02:24+00:00,2023-01-30 11:55:12+00:00
2,G550,37,0,3,2019-04-03 18:24:12+00:00,2023-04-06 10:34:33+00:00
3,GL6T,12,0,1,2021-05-17 10:55:18+00:00,2022-11-08 09:20:29+00:00
4,X,5,0,1,2019-11-29 12:31:39+00:00,2022-10-21 14:01:10+00:00


Другие сочетания полей тех же адресов во всей базе:


,icao24,registration,typecode,icaoaircrafttype,rows,go_arounds,first_landing,last_landing,airports
0,3d4bd7,NaN,X,X,5,0,2019-11-29 12:31:39+00:00,2022-10-21 14:01:10+00:00,"EDDB, EDDM"
1,424b5e,MHERI,G550,G550,17,0,2020-02-09 20:56:51+00:00,2023-04-06 10:34:33+00:00,"EGSS, EPWA, LEMD, LIMC, LOWW, LSGG, LSZH"
2,4911f5,CSDOU,GL6T,GL6T,12,0,2021-05-17 10:55:18+00:00,2022-11-08 09:20:29+00:00,"EDDB, EDDF, EGLL, LEMD, LSGG"
3,4a09ae,YRBMN,737,82R,457,2,2019-01-05 17:56:33+00:00,2022-09-04 20:06:12+00:00,"EDDF, EDDM, EGLL, EHAM, EIDW, EKCH, ESSA, LEMD..."
4,4d0206,LXPHS,G550,G550,10,0,2019-04-03 18:24:12+00:00,2022-02-15 16:20:40+00:00,"EDDB, EGSS, EHAM, LIMC, LSGG"
5,a660f6,N51VE,G-V,G-V,8,0,2019-03-19 11:02:24+00:00,2023-01-30 11:55:12+00:00,"EDDM, EGSS, EHAM, LEMD, LSZH"
6,c032a0,CFTEL,G550,G550,10,0,2021-10-04 14:10:28+00:00,2023-01-13 19:02:17+00:00,"EDDB, EDDM, EGSS, EIDW"


ICAO-адресов: 7
Рабочая копия проверена по SHA256 и не изменена.


In [2]:
# Читаем семь отложенных оценок; все 519 строк пока остаются без исправлений
assessment = pd.read_csv(ROOT / 'europe/reports/preparation/05_unresolved_aircraft_assessment.csv', keep_default_na=False)
# Проверяем ожидаемый охват и отсутствие примененных предложений
assert len(assessment) == 7 and assessment['rows'].sum() == 519
assert not assessment['applied'].any()
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(assessment)
print('Кандидаты существуют только в диагностическом отчете. Исправления не применены.')

,icao24,registration,source_code,rows,candidate_designator,candidate_description,evidence_status,historical_limit,applied
0,4a09ae,YRBMN,82R,457,B738,L2J,official_indexed_extract_only,document_date_not_established,False
1,a660f6,N51VE,G-V,8,GLF5,L2J,official_pdf_reviewed,2026_does_not_establish_2019_2023,False
2,4911f5,CSDOU,GL6T,12,GLEX,L2J,official_pdf_reviewed,2023_does_not_establish_2021_2022,False
3,4d0206,LXPHS,G550,10,GLF5,L2J,official_indexed_extract_only,2023_snapshot_original_pdf_unavailable,False
4,424b5e,MHERI,G550,17,,,no_official_aircraft_identity_confirmation,not_established,False
5,c032a0,CFTEL,G550,10,,,no_official_aircraft_identity_confirmation,not_established,False
6,3d4bd7,,X,5,,,registration_missing,not_established,False


Кандидаты существуют только в диагностическом отчете. Исправления не применены.


## Итог шага 6

Все 519 записей относятся к семи ICAO-адресам; других сочетаний кодов этих адресов в базе не найдено. Четыре борта имеют кандидатов с разными ограничениями подтверждения, три остаются без подтвержденного кандидата. Ни один найденный документ не подтверждает идентичность во все даты наблюдений группы.

Доступные официальные источники: [FAA NAARMO, июнь 2026, PDF-страница 170](https://www.faa.gov/sites/faa.gov/files/2026-06/NAARMO_US_IGA_RVSM_Approvals_Part_135_91_Section_3_0.pdf), [ANAC, отчет за 2023 год, PDF-страница 217](https://www.anac.pt/SiteCollectionDocuments/Informacao_Gestao/Relatorios_Gestao/Relatorio_de_Atividades_Gest%C3%A3o_e_Contas_2023.pdf). Сведения поискового индекса AACR и DAC Luxembourg отдельно помечены: полный оригинал не удалось получить.

Новые преобразования не выполнены. Прежде чем использовать кандидатов, отдельно согласуем допустимость неполного исторического подтверждения. Альтернатива – сохранить записи неразрешенными и перейти к диагностике C40.

## Шаг 7 – Временная диагностика C40

Проверяем все 7 984 254 записи рабочей копии после справочного обогащения. C40 – метаданные входа в ASMA, 40-мильную область вокруг опорной точки аэропорта; это не отметка входа на глиссаду ILS.

Первичный источник полей – [описание автора на Zenodo](https://zenodo.org/records/10210178). [Статья авторов, разделы 2.2–2.3](https://doi.org/10.59490/joas.2023.7219) объясняет разные источники: посадочные метаданные из OpenSky и ASMA из EUROCONTROL; сопоставление по дню, аэропорту и ICAO24. Эти ключи не гарантируют однозначности конкретной записи.

[EUROCONTROL/EASA AVENIR, раздел 5.4, PDF-страница 43](https://www.eurocontrol.int/sites/default/files/2025-09/avenir-wg-pillar-1-final-report.pdf) описывает переход от последнего пересечения 40 NM к первому входу в обновленной методике. Это допускает гипотезу повторного входа после ухода, но не устанавливает алгоритм формирования нашего файла. Публичный ноутбук автора использует готовую таблицу; код ее исходного сопоставления в нем не найден.

Отдельно проверяем C40 после первой попытки, после последней попытки и после посадки. Группы пересекаются: 42 записи входят в группу из 87, их нельзя складывать. Проверяем пропуски и первый элемент списка попыток относительно его минимума. Соседние записи того же адреса в окне ±24 часа служат только диагностическими кандидатами, а не правилом исправления. Все времена в UTC; допуск и временные сдвиги не применяются.

In [1]:
# Проверяем положение времени C40 относительно попыток и посадки
from pathlib import Path
import runpy
import json
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'europe/data/source/osn23_landings_merged.parquet.gzip').exists())
# Диагностика сканирует всю базу и сохраняет категории, строки и пары для разбора
module = runpy.run_path(str(ROOT / 'europe/scripts/diagnose_c40.py'))
c40_flags, c40_coverage, c40_records, c40_pairs = module['main']()
c40_journal = json.loads((ROOT / 'europe/reports/preparation/06_c40_diagnostics.json').read_text(encoding='utf8'))
# Показываем счетчики и подтверждение неизменности входных файлов
display(c40_flags)
display(pd.DataFrame(c40_journal['categories']))
print('Уникальных записей для разбора:', c40_journal['unique_review_rows'])
print('Исходник и рабочая копия неизменны:', c40_journal['source_unchanged'], c40_journal['input_unchanged'])

Проверено строк: 1000000


Проверено строк: 2000000


Проверено строк: 3000000


Проверено строк: 4000000


Проверено строк: 5000000


...

Проверено строк: 7000000


{
  "checked_date": "2026-10-08",
  "input": "europe\\data\\working\\02_aircraft_reference_enriched.parquet",
  "input_sha256": "25b6010d4d7197a22a41ee597fd3675bd325ae43107af6709d7c33f96f097187",
  "input_unchanged": true,
  "source_sha256": "1160bdbea6fa93249a00768452be64047c6d2461bb0843d5ecfc52e6942c7f02",
  "source_unchanged": true,
  "rows_checked": 7984254,
  "go_arounds": 26253,
  "rows_changed": 0,
  "rows_deleted": 0,
  "flags": [
    {
      "flag": "after_first_attempt",
      "rows": 87,
      "go_arounds": 45
    },
    {
      "flag": "after_landing",
      "rows": 42,
      "go_arounds": 0
    },
    {
      "flag": "after_last_attempt",
      "rows": 42,
      "go_arounds": 0
    },
    {
      "flag": "equal_first_attempt",
      "rows": 0,
      "go_arounds": 0
    },
    {
      "flag": "equal_landing",
      "rows": 0,
      "go_arounds": 0
    }
  ],
  "unique_review_rows": 87,
  "first_attempt_not_earliest_rows": 0,
  "null_attempt_lists": 0,
  "null_attempt_elemen

,flag,rows,go_arounds
0,after_first_attempt,87,45
1,after_landing,42,0
2,after_last_attempt,42,0
3,equal_first_attempt,0,0
4,equal_landing,0,0


,category,rows,go_arounds
0,after_first_before_or_equal_landing,45,45
1,after_landing,42,0
2,before_first_attempt,6331785,18361
3,missing_c40,1652382,7847


Уникальных записей для разбора: 87
Исходник и рабочая копия неизменны: True True


In [2]:
# Сводим записи по аэропортам и временным категориям C40
print('Распределение 87 записей по аэропортам и группам:')
display(c40_records.groupby(['airport', 'c40_time_category']).agg(
    rows=('GoA', 'size'), go_arounds=('GoA', 'sum'),
    min_landing_minus_c40_minutes=('landing_minus_c40_minutes', 'min'),
    max_landing_minus_c40_minutes=('landing_minus_c40_minutes', 'max')).reset_index())

# Выводим все случаи времени C40 после посадки с ключевыми временами
print('Все 42 записи с C40 после посадки:')
c40_review_columns = ['source_row_position', 'flight_id', 'icao24', 'airport', 'landing_time',
                     'first_attempt_time', 'last_attempt_time', 'C40_CROSS_TIME',
                     'GoA', 'n_attempts', 'landing_minus_c40_minutes']
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(c40_records.loc[c40_records['after_landing'], c40_review_columns])

Распределение 87 записей по аэропортам и группам:


,airport,c40_time_category,rows,go_arounds,min_landing_minus_c40_minutes,max_landing_minus_c40_minutes
0,EDDB,after_first_before_or_equal_landing,1,1,23.000000,23.000000
1,EDDF,after_first_before_or_equal_landing,6,6,12.733333,30.883333
2,EDDM,after_first_before_or_equal_landing,2,2,11.200000,14.100000
3,EGLL,after_landing,2,0,-5.133333,-0.433333
4,EGSS,after_landing,1,0,-2.333333,-2.333333
5,EHAM,after_first_before_or_equal_landing,2,2,17.533333,21.966667
6,EHAM,after_landing,1,0,-1.633333,-1.633333
7,EKCH,after_landing,1,0,-6.616667,-6.616667
8,LEMD,after_first_before_or_equal_landing,16,16,11.550000,27.083333
9,LEMD,after_landing,3,0,-9.200000,-2.433333


Все 42 записи с C40 после посадки:


,source_row_position,flight_id,icao24,airport,landing_time,first_attempt_time,last_attempt_time,C40_CROSS_TIME,GoA,n_attempts,landing_minus_c40_minutes
0,696773,VLG6273_23112,3451d3,LIRF,2019-04-19 22:05:09+00:00,2019-04-19 22:05:09+00:00,2019-04-19 22:05:09+00:00,2019-04-19 22:12:02+00:00,False,1,-6.883333
1,916854,VLG80EU_15642,34538d,LFPO,2019-05-21 20:30:18+00:00,2019-05-21 20:30:18+00:00,2019-05-21 20:30:18+00:00,2019-05-21 20:34:02+00:00,False,1,-3.733333
2,1223930,IBS37HF_10828,341682,LEMD,2019-07-02 20:52:51+00:00,2019-07-02 20:50:22+00:00,2019-07-02 20:50:22+00:00,2019-07-02 21:02:03+00:00,False,1,-9.200000
3,1474355,EXS1458_14252,4070ea,EGSS,2019-08-05 13:05:45+00:00,2019-08-05 13:05:45+00:00,2019-08-05 13:05:45+00:00,2019-08-05 13:08:05+00:00,False,1,-2.333333
4,1477978,ABR6EC_178801,4ca282,LFPG,2019-08-05 22:06:17+00:00,2019-08-05 22:04:27+00:00,2019-08-05 22:04:27+00:00,2019-08-05 22:14:18+00:00,False,1,-8.016667
5,1722891,ANE15HD_99479,345543,LEMD,2019-09-07 08:26:24+00:00,2019-09-07 08:25:59+00:00,2019-09-07 08:25:59+00:00,2019-09-07 08:28:50+00:00,False,1,-2.433333
6,2004968,EZS65DW_62153,4b1a65,LIRF,2019-10-15 18:32:18+00:00,2019-10-15 18:30:09+00:00,2019-10-15 18:30:09+00:00,2019-10-15 18:40:50+00:00,False,1,-8.533333
7,2598399,DAH1144_4589,0a0077,LFPG,2020-01-13 14:49:56+00:00,2020-01-13 14:49:56+00:00,2020-01-13 14:49:56+00:00,2020-01-13 14:50:44+00:00,False,1,-0.800000
8,2600997,ABR6BN_82206,4ca282,LFPG,2020-01-13 22:53:47+00:00,2020-01-13 22:53:28+00:00,2020-01-13 22:53:28+00:00,2020-01-13 22:56:21+00:00,False,1,-2.566667
10,2766525,EZS52QB_28471,4b1a2a,LSGG,2020-02-08 20:45:50+00:00,2020-02-08 20:45:46+00:00,2020-02-08 20:45:46+00:00,2020-02-08 20:50:30+00:00,False,1,-4.666667


In [3]:
# Выделяем C40 после первой попытки, исключая случаи после посадки
print('Все 45 записей с C40 после первой попытки, но до посадки:')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(c40_records.loc[c40_records['after_first_attempt'] & ~c40_records['after_landing'], c40_review_columns])
# Проверяем повторное использование одной отметки у того же борта и аэропорта
print('Повторное присоединение той же отметки C40 к другому наблюдению того же адреса и аэропорта:')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(c40_pairs.loc[c40_pairs['same_airport'] & c40_pairs['same_c40_time']])
print('Это свидетельство неоднозначности сопоставления; исправления и объединение записей не выполнены.')

Все 45 записей с C40 после первой попытки, но до посадки:


,source_row_position,flight_id,icao24,airport,landing_time,first_attempt_time,last_attempt_time,C40_CROSS_TIME,GoA,n_attempts,landing_minus_c40_minutes
9,2734305,TVF4511_19553,3964f5,LFPO,2020-02-03 21:40:50+00:00,2020-02-03 21:03:23+00:00,2020-02-03 21:40:37+00:00,2020-02-03 21:20:13+00:00,True,2,20.616667
11,2771720,OMA131_87474,70c0ef,LFPG,2020-02-09 19:08:56+00:00,2020-02-09 18:26:40+00:00,2020-02-09 19:08:56+00:00,2020-02-09 18:51:25+00:00,True,2,17.516667
12,2771854,TRA4H_64916,4853d2,EHAM,2020-02-09 19:30:15+00:00,2020-02-09 18:44:34+00:00,2020-02-09 19:30:15+00:00,2020-02-09 19:08:17+00:00,True,2,21.966667
14,3000179,TVF93PC_49784,3964f7,LEMD,2020-03-15 18:52:59+00:00,2020-03-15 18:24:15+00:00,2020-03-15 18:52:59+00:00,2020-03-15 18:39:59+00:00,True,2,13.000000
17,3237730,DLH1UP_49736,3c65d0,EDDF,2020-08-13 06:38:03+00:00,2020-08-13 05:57:23+00:00,2020-08-13 06:37:44+00:00,2020-08-13 06:07:10+00:00,True,3,30.883333
21,3519971,SWT2KT_6372,344485,LFPG,2020-11-11 10:24:28+00:00,2020-11-11 08:19:41+00:00,2020-11-11 10:24:28+00:00,2020-11-11 09:04:30+00:00,True,2,79.966667
23,3615225,SXS5FC_87009,4bcde4,EHAM,2020-12-27 07:54:45+00:00,2020-12-27 06:54:23+00:00,2020-12-27 07:54:25+00:00,2020-12-27 07:37:13+00:00,True,3,17.533333
24,3615785,AUI3FU_86756,508371,LFPG,2020-12-27 11:58:28+00:00,2020-12-27 11:04:59+00:00,2020-12-27 11:58:23+00:00,2020-12-27 11:41:43+00:00,True,2,16.750000
25,3762911,UAE9993_120041,89616f,EDDF,2021-03-13 11:05:33+00:00,2021-03-13 10:37:06+00:00,2021-03-13 11:05:12+00:00,2021-03-13 10:47:19+00:00,True,2,18.233333
26,3904680,DLH46J_24768,3c5462,EDDF,2021-05-16 18:14:07+00:00,2021-05-16 17:20:27+00:00,2021-05-16 18:13:40+00:00,2021-05-16 17:46:07+00:00,True,2,28.000000


Повторное присоединение той же отметки C40 к другому наблюдению того же адреса и аэропорта:


,flagged_row_position,related_row_position,icao24,flagged_airport,related_airport,same_airport,same_c40_time,same_c40_lat_lon,same_callsign,related_landing_minus_c40_minutes,landing_difference_minutes
17,1223930,1224005,341682,LEMD,LEMD,True,True,True,True,10.233333,19.433333
23,1477978,1478031,4ca282,LFPG,LFPG,True,True,True,True,11.466667,19.483333
33,2004968,2005110,4b1a65,LIRF,LIRF,True,True,True,True,11.266667,19.800000


Это свидетельство неоднозначности сопоставления; исправления и объединение записей не выполнены.


## Итог шага 7 – Правило очистки пока не применено

Полный проход подтвердил аудит 01: 87 записей имеют C40 после первой попытки. В их составе 45 отмеченных уходов с 2–3 попытками: C40 предшествует последней попытке и посадке. Такие случаи совместимы с повторным входом, но требуют траекторий для подтверждения. Эта отметка C40 не подходит как момент прогноза до первой попытки; переносить ее назад искусственно нельзя.

Другие 42 записи имеют одну попытку, GoA=False и C40 через 26 секунд – 9 минут 12 секунд после указанной посадки. Это внутреннее противоречие метаданных. В 23 случаях аэропорт – LSGG; концентрация не является основанием исключить весь аэропорт. Три записи имеют тот же C40 и координаты, что более позднее наблюдение того же адреса, аэропорта и позывного. Это может отражать сопоставление или сегментацию; определить ошибочное поле и исправить метку ухода по одной таблице нельзя.

C40 отсутствует в 1 652 382 записях, включая 7 847 уходов. Пропуски не приравниваем к противоречиям. В LTFJ C40 отсутствует во всех записях; статья объясняет отсутствие покрытия в EU-NM. Для остальных аэропортов не переносим проценты покрытия статьи на наш полный файл.

Предлагаем следующим согласованным действием добавить только диагностические флаги в отдельную рабочую копию: C40 отсутствует; C40 после первой попытки; C40 после посадки. Все записи и исходные поля сохранить. Для будущего анализа отдельно определить допустимость C40 относительно момента прогноза. Обнуление C40, исключение строк, изменение GoA, объединение сегментов и исключение аэропортов сейчас не выполняются.

Временные диагностические флаги используют сведения завершенного наблюдения и не предназначены для подачи в модель раннего прогноза. Отбор по ним может изменить состав целевых классов и требует отдельной оценки.

Подробный [разбор C40](../reports/preparation/06_c40_review.md) содержит ограничения и источники. Проверены SHA256 исходника и рабочей копии до и после проходов; изменений нет.

## Шаг 8 – Согласованное добавление диагностических флагов C40

Добавляем в новую копию `europe/data/working/03_c40_diagnostic_flags.parquet` только три булевых поля. Все записи и существующие значения сохраняются. Правила основаны на полном разборе шага 7 и его [первичных источниках](../reports/preparation/06_c40_review.md), а не на новых авиационных порогах.

| Поле | True означает |
| --- | --- |
| `c40_time_missing` | В C40_CROSS_TIME физический null. |
| `c40_after_first_attempt` | C40 строго позже первого элемента attempt_times. |
| `c40_after_landing` | C40 строго позже landing_time. |

Флаги не содержат null. Если C40 отсутствует, оба временных флага False: это не означает подтвержденную корректность C40. Сравнения выполняются в UTC, без допуска и искусственного сдвига времени. Первый элемент списка проверяется на равенство минимальному времени; пустые списки и null-элементы не допускаются для этого проверенного входа.

Это диагностические поля. Временные флаги используют сведения завершенного наблюдения и не являются входными признаками раннего прогноза. Источник и входная копия проверяются по SHA256 до и после. После записи сравниваем каждое прежнее значение, независимо пересчитываем флаги через NumPy и сверяем счетчики с шагом 7.

In [1]:
# Создаем согласованную копию с тремя диагностическими флагами C40
from pathlib import Path
import runpy
import json
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'europe/data/source/osn23_landings_merged.parquet.gzip').exists())
# Скрипт сохраняет исходные поля и независимо проверяет флаги каждой строки
module = runpy.run_path(str(ROOT / 'europe/scripts/apply_c40_flags.py'))
c40_changes, c40_combinations = module['main']()
display(c40_changes)
print('Взаимоисключающие сочетания флагов:')
display(c40_combinations)
# Читаем журнал преобразования и подтверждаем сохранение строк и меток
c40_applied = json.loads((ROOT / 'europe/reports/preparation/07_c40_flags_applied.json').read_text(encoding='utf8'))
print('Новая рабочая копия:', c40_applied['output'])
print('Записей до и после:', c40_applied['rows_before'], c40_applied['rows_after'])
print('Отмеченных уходов до и после:', c40_applied['go_arounds_before'], c40_applied['go_arounds_after'])
print('Все существующие поля неизменны:', c40_applied['existing_columns_unchanged'])
print('Флаги независимо проверены для каждой строки:', c40_applied['independent_per_row_flag_validation'])

Добавлены флаги, строк: 1000000


Добавлены флаги, строк: 2000000


Добавлены флаги, строк: 3000000


Добавлены флаги, строк: 4000000


Добавлены флаги, строк: 5000000


...

Сохранены все записи и существующие поля: 7984254


,field,true_rows,false_rows,null_rows,go_arounds_with_true
0,c40_time_missing,1652382,6331872,0,7847
1,c40_after_first_attempt,87,7984167,0,45
2,c40_after_landing,42,7984212,0,0


Взаимоисключающие сочетания флагов:


,c40_time_missing,c40_after_first_attempt,c40_after_landing,rows,go_arounds
0,False,False,False,6331785,18361
1,True,False,False,1652382,7847
2,False,True,False,45,45
3,False,True,True,42,0


Новая рабочая копия: europe\data\working\03_c40_diagnostic_flags.parquet
Записей до и после: 7984254 7984254
Отмеченных уходов до и после: 26253 26253
Все существующие поля неизменны: True
Флаги независимо проверены для каждой строки: True


## Итог шага 8

Сохранены все 7 984 254 записи и 26 253 отмеченных ухода. Добавлены флаги: C40 отсутствует – 1 652 382 записи; после первой попытки – 87; после посадки – 42. Последние 42 входят в 87, а не образуют дополнительную группу. У 45 оставшихся записей с флагом после первой попытки отмечен уход; исправления этих случаев не выполнялись.

Неизменность всех прежних полей проверена полным сравнением. Исходный Parquet и предыдущая рабочая копия сохранены. Новые флаги не меняют тип самолета, время или GoA и не исключают аэропорты. [Журнал преобразования](../reports/preparation/07_c40_flags_applied.json) хранит определения, счетчики и контрольные суммы.

Следующий разбор – повторяющиеся flight_id: отличаем повтор идентификатора от дубля наблюдения и возможного разделения одного полета на сегменты. Новое правило удаления или объединения заранее не задано.

## Шаг 9 – Повторяющиеся flight_id

Читаем последнюю копию `03_c40_diagnostic_flags.parquet`. Сначала считаем идентификаторы по всему файлу, затем извлекаем все записи повторяющихся значений. Сравниваем UTC-даты, ICAO24, аэропорты, позывные, времена и содержательные поля. Проверяем все пары внутри каждой группы, включая возможные совпадения в один день; примерная голова файла не используется для общих выводов.

В [документации разработчиков traffic](https://traffic-viz.github.io/api_reference/traffic.core.traffic.html#traffic.core.Traffic.assign_id) идентификатор может строиться из позывного и последовательного индекса. [Flight.assign_id](https://traffic-viz.github.io/api_reference/traffic.core.flight.html#traffic.core.Flight.assign_id) описывает шаблон `{self.callsign}_{idx:>03}`. Сходство нашего формата – подсказка, а не доказательство алгоритма автора за 2019–2023 годы. Сохраненный публичный ноутбук автора читает готовый Parquet и не объясняет создание этих идентификаторов.

Повтор flight_id не равен дублю посадки. Отдельно сравниваем все поля рабочей копии без служебного индекса и набор исходных содержательных полей после нормализации. Ключ `landing_time + icao24 + airport` проверен в аудите 01; его уникальность в файле не доказывает идентичность физического полета. Все источники и текущая копия проверяются по SHA256.

In [1]:
# Исследуем повторяющиеся flight_id, не удаляя наблюдения
from pathlib import Path
import runpy
import json
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'europe/data/source/osn23_landings_merged.parquet.gzip').exists())
# Скрипт сравнивает даты, адреса, аэропорты и точные значения внутри групп
module = runpy.run_path(str(ROOT / 'europe/scripts/diagnose_flight_ids.py'))
id_groups, id_categories, id_examples, id_near_pairs = module['main']()
id_journal = json.loads((ROOT / 'europe/reports/preparation/08_flight_id_diagnostics.json').read_text(encoding='utf8'))
# Показываем размеры групп и отличаем коллизии имен от точных дублей
display(id_categories)
display(pd.read_csv(ROOT / 'europe/reports/preparation/08_flight_id_group_sizes.csv'))
print('Повторяющихся идентификаторов:', id_journal['repeated_identifiers'])
print('Записей в этих группах:', id_journal['rows_with_repeated_identifier'])
print('Превышение числа записей над числом идентификаторов:', id_journal['excess_identifier_rows'])
print('Точных дублей рабочей копии в группах:', id_journal['exact_working_duplicate_excess_in_repeated_groups'])
print('Групп с разными адресами:', id_journal['groups_with_multiple_aircraft_addresses'])
print('Групп с разными аэропортами:', id_journal['groups_with_multiple_airports'])
print('Пар одного адреса и аэропорта в одну UTC-дату:', id_journal['same_day_same_address_airport_pairs'])

Проверено строк: 1000000


Проверено строк: 2000000


Проверено строк: 3000000


Проверено строк: 4000000


Проверено строк: 5000000


...

Проверено строк: 7000000


{
  "input": "europe\\data\\working\\03_c40_diagnostic_flags.parquet",
  "input_sha256": "ddf11438bab44137242ffcf4da928418e116efd46defce58dc8a388b67942b6d",
  "source_sha256": "1160bdbea6fa93249a00768452be64047c6d2461bb0843d5ecfc52e6942c7f02",
  "input_unchanged": true,
  "source_unchanged": true,
  "rows_checked": 7984254,
  "rows_changed": 0,
  "rows_deleted": 0,
  "repeated_identifiers": 23889,
  "rows_with_repeated_identifier": 53127,
  "excess_identifier_rows": 29238,
  "go_arounds_in_repeated_identifier_rows": 121,
  "max_rows_per_identifier": 7,
  "exact_working_duplicate_excess_in_repeated_groups": 0,
  "normalized_source_business_duplicate_excess_in_repeated_groups": 0,
  "core_key_duplicate_excess_in_repeated_groups": 0,
  "groups_with_multiple_aircraft_addresses": 22547,
  "groups_with_multiple_airports": 1667,
  "groups_with_multiple_callsigns_including_null": 0,
  "groups_with_at_least_one_go_around": 121,
  "groups_with_mixed_go_around_labels": 121,
  "groups_with_multipl

,category,identifier_groups,rows,go_arounds
0,different_dates_only,23889,53127,121


,rows_per_identifier,identifier_groups,go_arounds
0,2,20069,89
1,3,2594,14
2,4,925,13
3,5,300,5
4,7,1,0


Повторяющихся идентификаторов: 23889
Записей в этих группах: 53127
Превышение числа записей над числом идентификаторов: 29238
Точных дублей рабочей копии в группах: 0
Групп с разными адресами: 22547
Групп с разными аэропортами: 1667
Пар одного адреса и аэропорта в одну UTC-дату: 0


In [2]:
# Разбираем примеры коллизий flight_id с полным контекстом
print('Полные примеры групп повторяющихся идентификаторов:')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(id_examples)
print('Записей с отмеченным уходом в повторяющихся группах:', id_journal['go_arounds_in_repeated_identifier_rows'])
print('Групп со смешанными GoA:', id_journal['groups_with_mixed_go_around_labels'])
# Оцениваем потери целевого класса при гипотетическом удалении по имени
print('Если оставить первую строку каждого flight_id, потеряем отмеченных уходов:',
      id_journal['go_arounds_lost_if_deduplicating_flight_id_keep_first_in_file_order'])
print('Это расчет последствий возможного правила; строки не удалялись.')

Полные примеры групп повторяющихся идентификаторов:


,flight_id,source_row_position,icao24,airport,callsign,landing_date_utc,landing_time,first_attempt_time,GoA,n_attempts,c40_after_landing
8575,ETH500_129,1968662,040030,EIDW,ETH500,2019-10-11,2019-10-11 04:18:08+00:00,2019-10-11 04:18:08+00:00,False,1,False
15794,ETH500_129,3040984,040046,EIDW,ETH500,2020-04-16,2020-04-16 03:55:29+00:00,2020-04-16 03:55:29+00:00,False,1,False
25752,ETH500_129,3905261,040030,EIDW,ETH500,2021-05-17,2021-05-17 04:10:48+00:00,2021-05-17 04:10:48+00:00,False,1,False
35500,ETH500_129,5096302,04002f,EIDW,ETH500,2022-01-23,2022-01-23 04:15:10+00:00,2022-01-23 04:14:53+00:00,False,1,False
51197,ETH500_129,7702038,040032,EIDW,ETH500,2023-04-11,2023-04-11 04:34:23+00:00,2023-04-11 04:34:06+00:00,False,1,False
8710,ETH500_130,1998882,040030,EIDW,ETH500,2019-10-15,2019-10-15 03:31:14+00:00,2019-10-15 03:31:14+00:00,False,1,False
15861,ETH500_130,3046418,040046,EIDW,ETH500,2020-04-25,2020-04-25 03:49:28+00:00,2020-04-25 03:49:26+00:00,False,1,False
25781,ETH500_130,3909981,040030,EIDW,ETH500,2021-05-19,2021-05-19 03:55:36+00:00,2021-05-19 03:55:36+00:00,False,1,False
35558,ETH500_130,5105047,04002f,EIDW,ETH500,2022-01-25,2022-01-25 03:44:25+00:00,2022-01-25 03:44:25+00:00,False,1,False
51305,ETH500_130,7718398,040032,EIDW,ETH500,2023-04-14,2023-04-14 03:46:23+00:00,2023-04-14 03:46:09+00:00,False,1,False


Записей с отмеченным уходом в повторяющихся группах: 121
Групп со смешанными GoA: 121
Если оставить первую строку каждого flight_id, потеряем отмеченных уходов: 71
Это расчет последствий возможного правила; строки не удалялись.


In [3]:
# Проверяем, различает ли UTC-дата строки внутри каждого повторяющегося имени
print('Группы с повтором даты или контекста в один день:')
display(id_groups.loc[~id_groups['date_alone_disambiguates_identifier']])
# Показываем близкие пары и подтверждаем отсутствие изменений базы
display(id_near_pairs)
print('Исходник и рабочая копия неизменны:', id_journal['source_unchanged'], id_journal['input_unchanged'])
print('Изменено / удалено строк:', id_journal['rows_changed'], id_journal['rows_deleted'])

Группы с повтором даты или контекста в один день:


,flight_id,rows,go_arounds,dates,years,addresses,airports,callsigns,first_landing,last_landing,c40_after_landing_rows,landing_span_days,date_alone_disambiguates_identifier,same_day_address_airport_repeat,category


,flight_id,source_row_position_a,icao24_a,airport_a,callsign_a,landing_date_utc_a,landing_time_a,first_attempt_time_a,GoA_a,n_attempts_a,...,first_attempt_time_b,GoA_b,n_attempts_b,same_icao24,same_airport,same_callsign,same_landing_date_utc,same_landing_time,landing_gap_minutes,attempt_to_landing_windows_overlap


Исходник и рабочая копия неизменны: True True
Изменено / удалено строк: 0 0


## Итог шага 9

23 889 значений flight_id повторяются в 53 127 записях. Все записи внутри каждого такого идентификатора относятся к разным UTC-датам. В 22 547 группах различаются ICAO24; в 1 667 – аэропорты. Совпадение имени не связывает их в один физический полет. Среди этих записей нет точных содержательных дублей или повторов ключа посадки.

121 группа содержит ровно одну запись GoA=True и другие записи GoA=False. Удаление по flight_id может уничтожить реальные наблюдения и изменить состав целевого класса. Идентификатор нельзя использовать как единственный ключ присоединения меток, попыток или траекторий; иначе возможны связи между разными датами и бортами. Совпадение `flight_id + UTC-дата посадки` различает записи текущего файла, но не гарантирует уникальность будущих данных и не заменяет проверку траекторного сопоставления, особенно около полуночи.

Решение: все строки сохраняем, исходный flight_id не переписываем. При подготовке связей используем проверенный составной ключ наблюдения и контекст времени, адреса и аэропорта. Отдельный идентификатор записи можно добавить следующим согласованным правилом; он будет служебным, а не признаком модели. Идентификатор физического полета потребует иной проверки.

Подробный [разбор повторов](../reports/preparation/08_flight_id_review.md) содержит ограничения и первичные источники. Следующий диагностический этап – записи с большим числом попыток; они не считаются ошибочными или учебными автоматически.

## Шаг 10 – записи с большим числом попыток

Проверяем весь файл, затем отдельно разбираем n_attempts > 2. Этот порог служит диагностике, а не авиационной норме. [Описание набора](https://zenodo.org/records/10210178) определяет число обнаруженных подходов; GoA отмечает наличие хотя бы одного ухода. Число n_attempts - 1 не принимаем за подтвержденное число эксплуатационных уходов. ILS содержит обозначения ВПП, а не доказательство фактического применения системы ILS.

Сохраняем времена, интервалы и исходные строки. Длительность между первой и последней попытками не является полной длительностью полета. Проверяем контрольные суммы исходника и рабочей копии.

In [1]:
# Готовим разбор нескольких попыток и интервальных характеристик
from pathlib import Path
import json
import runpy
import pandas as pd
region = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd() / 'europe'
# Скрипт проверяет все строки, сохраняет группу n_attempts > 2 и контрольные суммы
attempt_module = runpy.run_path(str(region / 'scripts/diagnose_attempts.py'))
attempt_counts, attempt_airports, high_attempts, attempt_intervals = attempt_module['main']()
attempt_journal = json.loads((region / 'reports/preparation/09_attempts_diagnostics.json').read_text(encoding='utf8'))
# Сравниваем число попыток и показываем структурные противоречия
display(attempt_counts)
display(pd.Series(attempt_journal['structural_flags'], name='Число противоречий'))

Проверено строк: 1000000


Проверено строк: 2000000


Проверено строк: 3000000


Проверено строк: 4000000


Проверено строк: 5000000


Проверено строк: 6000000


Проверено строк: 7000000


{
  "input": "europe\\data\\working\\03_c40_diagnostic_flags.parquet",
  "input_sha256": "ddf11438bab44137242ffcf4da928418e116efd46defce58dc8a388b67942b6d",
  "source_sha256": "1160bdbea6fa93249a00768452be64047c6d2461bb0843d5ecfc52e6942c7f02",
  "input_unchanged": true,
  "source_unchanged": true,
  "rows_checked": 7984254,
  "rows_changed": 0,
  "rows_deleted": 0,
  "diagnostic_definition": "n_attempts > 2; not an aviation limit",
  "high_attempt_rows": 1052,
  "two_attempt_rows": 25201,
  "go_arounds": 26253,
  "high_attempt_percent_of_go_around_rows": 4.007161086352036,
  "max_attempts": 33,
  "high_attempt_airports": 20,
  "high_attempt_aircraft_addresses": 361,
  "structural_flags": {
    "attempt_count_nonpositive": 0,
    "attempt_length_mismatch": 0,
    "ILS_length_mismatch": 0,
    "GoA_differs_from_more_than_one_attempt": 0,
    "null_attempt_element": 0,
    "descending_attempt_times": 0,
    "repeated_adjacent_attempt_time": 0,
    "last_attempt_after_landing": 0
  },
  "h

,n_attempts,rows,aircraft_addresses,airports,span_min_minutes,span_median_minutes,span_max_minutes,rows_same_runway,c40_missing_rows
0,2,25201,7575,20,0.200000,13.366667,171.383333,21681,6946
1,3,421,295,20,9.316667,30.683333,241.933333,268,277
2,4,104,53,18,11.233333,52.541667,211.266667,64,99
3,5,81,38,17,16.916667,62.050000,270.516667,40,80
4,6,69,29,16,27.150000,86.783333,233.066667,36,69
5,7,38,15,11,38.400000,104.866667,195.800000,18,38
6,8,21,11,12,45.583333,113.483333,250.533333,7,21
7,9,24,16,11,50.700000,104.583333,259.133333,6,23
8,10,19,11,10,61.983333,119.683333,230.950000,9,19
9,11,17,10,9,98.550000,146.450000,230.300000,4,17


attempt_count_nonpositive                 0
attempt_length_mismatch                   0
ILS_length_mismatch                       0
GoA_differs_from_more_than_one_attempt    0
null_attempt_element                      0
descending_attempt_times                  0
repeated_adjacent_attempt_time            0
last_attempt_after_landing                0
Name: Число противоречий, dtype: int64

In [2]:
# Показываем распределение по аэропортам и пять максимальных чисел попыток
display(attempt_airports)
with pd.option_context('display.max_columns', None):
    display(high_attempts.nlargest(5, 'n_attempts'))
# Различаем медиану отдельных промежутков и медиану окна по наблюдениям
print('Медиана интервала, минуты:', attempt_journal['high_attempt_gap_minutes']['median'])
print('Медиана окна первой–последней попытки, минуты:', attempt_journal['high_attempt_span_minutes']['median'])

,airport,rows,aircraft_addresses,max_attempts,median_span_minutes
0,EDDB,83,32,20,72.250000
1,EDDF,101,30,25,172.666667
2,EDDM,61,12,23,155.666667
3,EGCC,14,7,13,97.758333
4,EGLL,18,17,3,32.141667
5,EGSS,33,16,12,33.616667
6,EHAM,80,38,28,26.008333
7,EIDW,137,19,15,58.233333
8,EKCH,25,7,28,116.666667
9,EPWA,64,19,24,67.591667


,source_row_position,flight_id,icao24,callsign,registration,airport,landing_time,year,n_attempts,GoA,typecode,icaoaircrafttype,market_segment,first_attempt_time,last_attempt_time,first_to_last_attempt_minutes,first_attempt_to_landing_minutes,last_attempt_to_landing_seconds,min_attempt_gap_minutes,median_attempt_gap_minutes,max_attempt_gap_minutes,unique_runway_designations,runway_switches,c40_time_missing,c40_after_first_attempt,c40_after_landing
25877,7875471,FCK1EL_3815,3cce70,FCK1EL,DCFME,LOWW,2023-05-12 02:35:51+00:00,2023,33,True,B350,L2T,NaN,2023-05-11 22:19:01+00:00,2023-05-12 02:35:47+00:00,256.766667,256.833333,4.0,4.966667,6.100000,34.333333,1,0,True,False,False
10468,3378043,FCK511_5650,3cce71,FCK511,NaN,LOWW,2020-09-23 00:54:10+00:00,2020,31,True,NaN,NaN,NaN,2020-09-22 21:04:27+00:00,2020-09-23 00:53:58+00:00,229.516667,229.716667,12.0,4.816667,5.458333,27.283333,2,2,True,False,False
10478,3381024,FCK511_5651,3cce71,FCK511,NaN,LOWW,2020-09-23 23:59:58+00:00,2020,30,True,NaN,NaN,NaN,2020-09-23 20:40:40+00:00,2020-09-23 23:59:36+00:00,198.933333,199.300000,22.0,4.733333,5.883333,14.683333,2,3,True,False,False
12081,3828528,FCK511_4865,3cce71,FCK511,NaN,LOWW,2021-04-12 15:26:51+00:00,2021,30,True,NaN,NaN,Other,2021-04-12 10:44:18+00:00,2021-04-12 15:26:49+00:00,282.516667,282.550000,2.0,5.533333,6.483333,69.533333,2,1,True,False,False
18736,5735838,FCK311_8302,3cce70,FCK311,DCFME,LOWW,2022-05-17 01:49:50+00:00,2022,29,True,B350,L2T,NaN,2022-05-16 22:01:31+00:00,2022-05-17 01:49:24+00:00,227.883333,228.316667,26.0,3.116667,5.933333,23.150000,2,1,True,False,False


Медиана интервала, минуты: 9.016666666666667
Медиана окна первой–последней попытки, минуты: 60.875


In [3]:
# Короткие промежутки и полные последовательности требуют траекторного разбора
with pd.option_context('display.max_columns', None):
    display(high_attempts.nsmallest(5, 'min_attempt_gap_minutes'))
    display(pd.read_csv(region / 'reports/preparation/09_high_attempt_examples.csv'))
# Проверяем пересечение с проблемами C40 и сохранность исходных данных
print('C40 после первой попытки:', attempt_journal['high_attempt_c40_after_first_rows'])
print('Исходник и рабочая копия неизменны:', attempt_journal['source_unchanged'], attempt_journal['input_unchanged'])
print('Изменено / удалено строк:', attempt_journal['rows_changed'], attempt_journal['rows_deleted'])

,source_row_position,flight_id,icao24,callsign,registration,airport,landing_time,year,n_attempts,GoA,typecode,icaoaircrafttype,market_segment,first_attempt_time,last_attempt_time,first_to_last_attempt_minutes,first_attempt_to_landing_minutes,last_attempt_to_landing_seconds,min_attempt_gap_minutes,median_attempt_gap_minutes,max_attempt_gap_minutes,unique_runway_designations,runway_switches,c40_time_missing,c40_after_first_attempt,c40_after_landing
12577,3954070,ZXP24_100524,485b2f,ZXP24,PHPXX,EHAM,2021-06-04 06:48:05+00:00,2021,3,True,NaN,NaN,NaN,2021-06-04 06:22:36+00:00,2021-06-04 06:47:40+00:00,25.066667,25.483333,25.0,0.650000,12.533333,24.416667,2,2,True,False,False
16714,5177528,ZXP24_145897,485b2f,ZXP24,PHPXX,EHAM,2022-02-10 19:29:20+00:00,2022,3,True,NaN,NaN,NaN,2022-02-10 17:57:53+00:00,2022-02-10 19:28:20+00:00,90.450000,91.450000,60.0,0.650000,45.225000,89.800000,2,2,True,False,False
23316,7157626,PGT1SC_148296,4bb84b,PGT1SC,TCNBK,EDDF,2022-12-12 09:31:11+00:00,2022,3,True,A20N,L2J,Lowcost,2022-12-12 09:17:54+00:00,2022-12-12 09:30:54+00:00,13.000000,13.283333,17.0,1.466667,6.500000,11.533333,1,0,False,False,False
20083,6131764,DLH419_20578,3c4b24,DLH419,DABYD,EDDF,2022-07-12 06:44:27+00:00,2022,3,True,B748,L4J,Mainline,2022-07-12 06:30:54+00:00,2022-07-12 06:44:27+00:00,13.550000,13.550000,0.0,2.433333,6.775000,11.116667,1,0,False,False,False
10306,3313179,PROVA20_4454,33fd75,PROVA20,NaN,LIMC,2020-09-03 15:40:46+00:00,2020,4,True,NaN,NaN,NaN,2020-09-03 15:15:51+00:00,2020-09-03 15:27:05+00:00,11.233333,24.916667,821.0,2.550000,2.600000,6.083333,1,0,True,False,False


,source_row_position,flight_id,icao24,airport,n_attempts,attempt_number_from,attempt_number_to,time_from,time_to,runway_from,runway_to,gap_minutes,same_runway
0,3378043,FCK511_5650,3cce71,LOWW,31,1,2,2020-09-22 21:04:27+00:00,2020-09-22 21:17:32+00:00,16,16,13.083333,True
1,3378043,FCK511_5650,3cce71,LOWW,31,2,3,2020-09-22 21:17:32+00:00,2020-09-22 21:24:25+00:00,16,16,6.883333,True
2,3378043,FCK511_5650,3cce71,LOWW,31,3,4,2020-09-22 21:24:25+00:00,2020-09-22 21:30:28+00:00,16,16,6.050000,True
3,3378043,FCK511_5650,3cce71,LOWW,31,4,5,2020-09-22 21:30:28+00:00,2020-09-22 21:36:07+00:00,16,16,5.650000,True
4,3378043,FCK511_5650,3cce71,LOWW,31,5,6,2020-09-22 21:36:07+00:00,2020-09-22 21:41:37+00:00,16,16,5.500000,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,7875471,FCK1EL_3815,3cce70,LOWW,33,28,29,2023-05-12 01:50:50+00:00,2023-05-12 01:56:56+00:00,11,11,6.100000,True
93,7875471,FCK1EL_3815,3cce70,LOWW,33,29,30,2023-05-12 01:56:56+00:00,2023-05-12 02:09:33+00:00,11,11,12.616667,True
94,7875471,FCK1EL_3815,3cce70,LOWW,33,30,31,2023-05-12 02:09:33+00:00,2023-05-12 02:15:53+00:00,11,11,6.333333,True
95,7875471,FCK1EL_3815,3cce70,LOWW,33,31,32,2023-05-12 02:15:53+00:00,2023-05-12 02:21:57+00:00,11,11,6.066667,True


C40 после первой попытки: 3
Исходник и рабочая копия неизменны: True True
Изменено / удалено строк: 0 0


## Итог шага 10

Выделено 1 052 записи (4,01% строк GoA=True), максимум 33 попытки. Противоречий длины списков, порядка времен и связи GoA с числом попыток не обнаружено. Это структурная согласованность, а не подтверждение каждой метки по траектории.

У записи с 33 попытками указан DCFME. [Оператор FCS](https://fcs.aero/en/about-fcs/) описывает D-CFME как самолет летных проверок. Эта информация поддерживает гипотезу о специальном назначении, но не устанавливает задачу конкретного полета 2023 года. Другие записи автоматически учебными или проверочными не считаем. Текущая документация traffic не доказывает точную историческую конфигурацию алгоритма.

Сохраняем все записи. Число попыток, итоговые времена и последовательность ВПП известны после события и требуют исключения из предсобытийных признаков. Подробности и отложенные решения: [разбор](../reports/preparation/09_attempts_review.md). Следующий шаг – свести результаты диагностики в журнал правил и определить состав наблюдения для моделирования до очистки аэропортов и строк.